# Day 9 — Solution: Joint Distributions & Two-Asset Portfolios

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
import os
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

## E1 — Var of a sum, independence

In [ ]:
rng = np.random.default_rng(3)
N = 100_000
x = rng.normal(0.0005, 0.012, N)
y = rng.normal(0.0003, 0.009, N)
print(f"Var(x+y) sim {(x + y).var():.10f}")
print(f"Varx+Vary   {x.var() + y.var():.10f}")
print(f"SD(x+y) {np.sqrt(x.var() + y.var()):.6f} vs σ₁+σ₂ {0.012 + 0.009}")

Adding variances: SD = √(σ₁²+σ₂²) = 1.50% vs naive σ₁+σ₂ = 2.1% — the
√n diversification law, witnessed at n=2. **Independent risks partially
cancel; SDs add in quadrature, never linearly.**

## E2 — activating the correlation term

Construction: y = ρ(σ₂/σ₁)x + σ₂√(1−ρ²)z — check Var(y) = σ₂²(ρ² + 1−ρ²)
= σ₂² ✓ and Cov(x,y) = ρσ₁σ₂ ✓.

In [ ]:
rng = np.random.default_rng(4)
N = 100_000
s1, s2 = 0.012, 0.009
x = rng.normal(0, s1, N)
for rho in [-1, -0.5, 0, 0.5, 1]:
    z = rng.normal(0, 1, N)
    y = rho * (s2 / s1) * x + s2 * np.sqrt(1 - rho ** 2) * z
    v = 0.5 ** 2 * s1 ** 2 + 0.5 ** 2 * s2 ** 2 + 2 * 0.5 * 0.5 * rho * s1 * s2
    print(f"ρ={rho:+.1f}: sim {np.sqrt((0.5*x + 0.5*y).std()):.6f} "
          f"formula {np.sqrt(v):.6f}")

At ρ=−1 the portfolio vol collapses to |σ₁−σ₂|/2 ≈ 0.15%; at ρ=+1 it's
the average 1.05%. **The covariance term spans the entire difference
between a hedge and a doubling-up.**

## E3 — the frontier on real data

In [ ]:
if DATA_SOURCE == "real":
    px = get_prices(["SPY", "TLT"], start="2010-01-01")
    r = px.pct_change().dropna()
else:
    px = synthetic_prices(n_days=3000, n_assets=2, seed=19, corr=0.2)
    px.columns = ["SPY", "TLT"]
    r = px.pct_change().dropna()
    r["TLT"] = -r["TLT"]   # synth needs corr>=0; negate to model SPY/TLT's negative link
s1, s2 = r["SPY"].std(), r["TLT"].std()
cov = r.cov().iloc[0, 1]

w_grid = np.linspace(0, 1, 101)
port_sd = np.sqrt((w_grid * s1) ** 2 + ((1 - w_grid) * s2) ** 2
                  + 2 * w_grid * (1 - w_grid) * cov)
w_min = cov / (cov + s2 ** 2 - s1 ** 2)   # analytic min-vol weight on SPY
print(f"σ_SPY {s1:.4f} σ_TLT {s2:.4f} ρ {cov/(s1*s2):+.2f}")
print(f"min-vol weight on SPY: {w_min:.2f} (vol {np.interp(w_min, w_grid, port_sd):.4f})")
plt.plot(w_grid, port_sd); plt.axvline(w_min, color="red", ls="--")
plt.xlabel("weight on SPY"); plt.ylabel("portfolio vol"); plt.show()

**Expected reasoning.** Because ρ < 1 strictly, the min-vol weight puts
*some* weight on the higher-vol asset — the curve is concave-smiling and
its minimum sits strictly inside. Equal-weight vol is below σ_SPY alone.
**Common mistake:** "min vol = 100% in the lower-vol asset" — only true
at ρ = 1.

## E4 — the optimal hedge

In [ ]:
b_grid = np.linspace(-1, 1, 401)
vars_ = [ (r["SPY"] + b * r["TLT"]).var() for b in b_grid ]
b_scan = b_grid[np.argmin(vars_)]
b_formula = -cov / r["TLT"].var()
print(f"optimal hedge weight on TLT: scan {b_scan:+.3f} vs -Cov/Var {b_formula:+.3f}")
print(f"unhedged vol {s1:.4f} -> hedged {np.sqrt((r['SPY'] + b_formula * r['TLT']).var()):.4f}")

Scan and formula agree to grid resolution; hedged vol is materially below
SPY's. b* = −Cov/Var is minus the regression slope of SPY on TLT — day 11
of module 01's projection, now wearing a portfolio hat. (And the hedge is
estimated: it inherits all of E3's sampling error — ρ̂ from T days has
SE ≈ 1/√T, so a 63-day hedge weight is a coin toss with extra steps.)

## E5 — the 2009 question (exemplar)

The spread's variance is Var(W) + Var(L) − 2ρ·σW·σL. At ρ = −0.1, the
cross term *subtracts* ~20% of the leg variance — the calm-regime spread
vol is far below either leg. In the crash-rebound, ρ jumps to +0.8: the
cross term *adds* ~160% of leg variance. The spread's risk roughly
doubles-to-triples in the exact window both legs are individually wild —
a "market-neutral" book sized on the calm-regime ρ is carrying ~3× the
assumed risk at the moment of the margin call. The formula didn't
change; ρ did. Hedging with a historical covariance is a bet that the
regime holds.